In [0]:
df = spark.read.option("header","true").csv(
    "s3://banking-data-platform-nilesh/bronze/transactions/"
)

display(df.limit(10))

Timestamp,From Bank,Account2,To Bank,Account4,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
2022/09/01 00:20,010,8000EBD30,010,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
2022/09/01 00:20,03208,8000F4580,001,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2022/09/01 00:00,03209,8000F4670,03209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
2022/09/01 00:02,012,8000F5030,012,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
2022/09/01 00:06,010,8000F5200,010,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0
2022/09/01 00:03,001,8000F5AD0,001,8000F5AD0,6162.44,US Dollar,6162.44,US Dollar,Reinvestment,0
2022/09/01 00:08,001,8000EBAC0,001,8000EBAC0,14.26,US Dollar,14.26,US Dollar,Reinvestment,0
2022/09/01 00:16,001,8000EC1E0,001,8000EC1E0,11.86,US Dollar,11.86,US Dollar,Reinvestment,0
2022/09/01 00:26,012,8000EC280,002439,8017BF800,7.66,US Dollar,7.66,US Dollar,Credit Card,0
2022/09/01 00:21,001,8000EDEC0,0211050,80AEF5310,383.71,US Dollar,383.71,US Dollar,Credit Card,0


In [0]:
df.printSchema()

root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: string (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: string (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: string (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: string (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: string (nullable = true)



In [0]:
df = spark.read.option("header","true").csv(
    "s3://banking-data-platform-nilesh/bronze/transactions/"
)

display(df.limit(5))

Timestamp,From Bank,Account2,To Bank,Account4,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
2022/09/01 00:20,010,8000EBD30,010,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
2022/09/01 00:20,03208,8000F4580,001,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2022/09/01 00:00,03209,8000F4670,03209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
2022/09/01 00:02,012,8000F5030,012,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
2022/09/01 00:06,010,8000F5200,010,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [0]:
for c in df.columns:
    df = df.withColumnRenamed(
        c,
        c.lower().replace(" ", "_")
    )

print(df.columns)

['timestamp', 'from_bank', 'account2', 'to_bank', 'account4', 'amount_received', 'receiving_currency', 'amount_paid', 'payment_currency', 'payment_format', 'is_laundering']


In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable(
      "banking_catalog.banking_schema.bronze_transactions"
  )

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM banking_catalog.banking_schema.bronze_transactions
""").show()

+--------+
|COUNT(*)|
+--------+
| 5078345|
+--------+



In [0]:
spark.sql("SHOW CATALOGS").show()

+---------------+
|        catalog|
+---------------+
|banking_catalog|
|           fmcg|
|        samples|
|         system|
|      workspace|
+---------------+



In [0]:
spark.sql("SHOW SCHEMAS IN banking_catalog").show()

+------------------+
|      databaseName|
+------------------+
|    banking_schema|
|           default|
|information_schema|
+------------------+



In [0]:
accounts_df = spark.read \
    .option("header","true") \
    .csv("s3://banking-data-platform-nilesh/bronze/accounts/")

In [0]:
for c in accounts_df.columns:
    accounts_df = accounts_df.withColumnRenamed(
        c,
        c.lower().replace(" ", "_")
    )

print(accounts_df.columns)

['bank_name', 'bank_id', 'account_number', 'entity_id', 'entity_name']


In [0]:
accounts_df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable(
      "banking_catalog.banking_schema.bronze_accounts"
  )

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM banking_catalog.banking_schema.bronze_accounts
""").show()

+--------+
|COUNT(*)|
+--------+
|  518581|
+--------+



In [0]:
spark.sql("""
SELECT *
FROM banking_catalog.banking_schema.bronze_accounts
LIMIT 5
""").show()

+--------------------+-------+--------------+---------+--------------------+
|           bank_name|bank_id|account_number|entity_id|         entity_name|
+--------------------+-------+--------------+---------+--------------------+
| Portugal Bank #4507| 331579|     80B779D80|80062E240|Sole Proprietorsh...|
|     Canada Bank #27|    210|     809D86900|800C998A0|  Corporation #33520|
|         UK Bank #33|  21884|     80812BE00|800C47F50|  Partnership #35397|
|  Germany Bank #4815|  32742|     81047F300|80096F0B0|  Corporation #48813|
|National Bank of ...| 127390|     80BD8CF00|800FB8760|    Corporation #889|
+--------------------+-------+--------------+---------+--------------------+

